# common_likely_pathogenic_variants.ipynb

Find variants enriched in ecDNA+ relative to ecDNA- samples.

In [ ]:
import cyvcf2
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
import os
import scipy.stats
from statsmodels.stats.multitest import multipletests

In [ ]:
def parse_CSQ(fields,csq):
    '''
    parse an input string into a nice dictionary.
    For CBTN somatic variant files, CSQ has 73 entries, separated by '|'. 
    Variants may have more than 1 CSQ associated with multiple transcripts, separated by ','.
    '''
    fields = fields.split('|')
    csqs = csq.split(',')
    csqs = [x.split('|') for x in csqs]
    csqs = list(zip(*csqs))
    csqs = [set(x) - {''} for x in csqs]
    #return csqs
    assert len(csqs) == len(fields),f'{len(csqs)},{len(fields)}'
    return dict(zip(fields,csqs))

In [ ]:
def parse_vcf(vcf_path):
    '''
    read a vcf file into a pandas dataframe.
    '''
    all_variants = pd.DataFrame()
    vcf = cyvcf2.VCF(vcf_path)
    csq_header = vcf.get_header_type('CSQ')['Description'].split('Format: ')[-1]
    for variant in vcf:
        values ={
            'REF':variant.REF, 
            'ALT':variant.ALT, 
            'CHROM':variant.CHROM, 
            'start':variant.start, 
            'end':variant.end, 
            'ID':variant.ID,
            'FILTER':variant.FILTER, 
            'QUAL':variant.QUAL,
            'HotSpot':variant.INFO.get('HotSpotAllele'),
            'CSQ':variant.INFO.get('CSQ')
        }
        csq = parse_CSQ(csq_header,values['CSQ'])
        #print(len(csq))
        values = csq | values
        #print(values)
        values = pd.DataFrame([values])
        keep = ['CHROM','start','end','REF','ALT','HotSpot','SYMBOL','Consequence','IMPACT','CLIN_SIG']
        values = values[keep]
        all_variants = pd.concat([all_variants,values])
    return all_variants

In [ ]:
def get_vcfs(directory="data/filter/likely_pathogenic"):
    for filename in os.listdir(directory):
        full_path = os.path.join(directory, filename)
        if os.path.isfile(full_path) and filename.endswith(".vcf.gz"):
            yield full_path
def extract_sample_name(filepath):
    sample = os.path.basename(filepath)
    sample = sample.split('.')[0]
    return sample

def get_all_pathogenic_variants(dry_run=False):
    '''
    Read all patient genomes,
    filter for pathogenic variants,
    and aggregate into a single pandas dataframe.
    '''
    big_df = pd.DataFrame()
    i=0
    for vcf in get_vcfs():
        i=i+1
        name = extract_sample_name(vcf)
        print('Now processing sample number',i,':',name)
        variants_df = parse_vcf(vcf)
        variants_df['sample'] = name
        big_df = pd.concat([big_df,variants_df])
        if dry_run == True:
            if i > 5:
                break
    # one row per gene affected
    big_df = big_df.explode('SYMBOL')
    return big_df

In [ ]:
likely_path_variants_df = get_all_pathogenic_variants(dry_run=False)

In [ ]:
def implode(df):
    return df.groupby(['sample'], as_index=True).agg({
        'SYMBOL': lambda x: set(x)
    })
likely_path_variants_df = implode(likely_path_variants_df)

In [ ]:
# how many samples with at least 1 pathogenic variant?
print(len(likely_path_variants_df))
# how many genes?

In [ ]:
# DEBUG
likely_path_variants_df[likely_path_variants_df.index.str.startswith('SJ')]

In [ ]:
def read_PBTA_manifest(file='./data/pedpancan/manifest_20250910_143948.csv'):
    df = pd.read_csv(file)
    df=df[df.name.str.endswith('.gz')]
    df['index'] = df.name.map(lambda x: os.path.basename(x).split('.')[0])
    df = df.set_index('index')
    df = df[['Kids First Biospecimen ID']].copy()
    return df

def read_sj_manifest(file='./data/pedpancan/somatic_vcfs.tsv'):
    df = pd.read_table(file)
    df = df[df['vcf_file_name'] != 'SJST030131_D3_G1.Somatic.vcf.gz']
    df.index = df.sample_name
    df["vcf_file_root"] = df["vcf_file_name"].str.split(".", n=1).str[0]
    return df

In [ ]:
# DEBUG 
read_sj_manifest().head(n=1)

In [ ]:
def read_biosample_table(file='./data/pedpancan/Supplementary Tables.xlsx'):
    return pd.read_excel(file,sheet_name='2. Biosamples',index_col='biosample_id')

In [ ]:
def merge_ecDNA_manifest_variants(ecDNA_df,manifest_df,variants_df):
    manifest_df['file_id'] = manifest_df.index
    manifest_df = manifest_df.set_index('Kids First Biospecimen ID')
    df = ecDNA_df.join(manifest_df,how='inner')
    df['biosample_id'] = df.index
    df = df.set_index('file_id')
    df = df.join(variants_df,how='left')
    df['SYMBOL'] = df['SYMBOL'].apply(lambda x: set() if pd.isna(x) else x)
    # filters
    df = df[df.in_unique_patient_set].copy() # drop biological replicates
    # TODO drop tumor types without ecDNA
    ec_tumor_types = df[df.amplicon_class.map(lambda x: 'ecDNA' in x)].cancer_type.unique()
    df = df[df.cancer_type.isin(ec_tumor_types)]
    return df

def sj_merge_ecDNA_manifest_variants(ecDNA_df,manifest_df,variants_df):
    df = ecDNA_df.join(manifest_df,how='inner')
    df.index = df["vcf_file_root"]
    df = df.join(variants_df,how='left')
    #df['SYMBOL'] = df['attr_subtype_biomarkers']
    df['SYMBOL'] = df['SYMBOL'].apply(lambda x: set() if pd.isna(x) else x)
    # filters
    df = df[df.in_unique_patient_set].copy() # drop biological replicates
    # TODO drop tumor types without ecDNA
    ec_tumor_types = df[df.amplicon_class.map(lambda x: 'ecDNA' in x)].cancer_type.unique()
    df = df[df.cancer_type.isin(ec_tumor_types)]
    return df

In [ ]:
# DEBUG
ecDNA_df = read_biosample_table()
manifest_df = read_sj_manifest()
df = ecDNA_df.join(manifest_df,how='inner')

df.head(n=1)

In [ ]:
# DEBUG
likely_path_variants_df[likely_path_variants_df.index.str.startswith('SJ')].head(n=1)

In [ ]:
df_PBTA = merge_ecDNA_manifest_variants(
    read_biosample_table(),
    read_PBTA_manifest(),
    likely_path_variants_df
)
df_PBTA

In [ ]:
df_SJ = sj_merge_ecDNA_manifest_variants(
    read_biosample_table(),
    read_sj_manifest(),
    likely_path_variants_df
)
df_SJ['file_id'] = df_SJ['vcf_file_id']
df_SJ.index = df_SJ['file_id']
df_SJ

In [ ]:
df_SJ.SYMBOL.astype(str).unique()

In [ ]:
df = pd.concat([df_PBTA, df_SJ], axis=0, ignore_index=False)
df

In [ ]:
len(df.index.unique())
len(df)

## Tests 
detect more common pathogenic variants in ecDNA+ samples by chi2 test, FDR correction. 

In [ ]:
def pathogenic_ccgc_table(file ='./data/databases/Cosmic_CancerGeneCensus_v101_GRCh38.tsv'):
    df = pd.read_table(file)
    df = df.loc[:, ['GENE_SYMBOL']]
    #df = df.set_index('GENE_SYMBOL')
    return df

In [ ]:
ccgc_df = pathogenic_ccgc_table()

In [ ]:
'''
Genes we don't care about:
MIR*
*-AS*
RNU6*
*-DT
'''
path_variant_genes = set(g for gene_set in df['SYMBOL'] for g in gene_set)
cancer_genes = set(ccgc_df['GENE_SYMBOL'].unique())
whitelist = {'H3-3A','H2AC4','H2AC5P','H2BC3','H3C2','H3C3','H4C2','HLA-B','HLA-A','AK4P1','CASC11','DNMT1','MIEN1','PRDM14','RHEB','RIT1','TGFBR1','TSPAN31'}

test_genes = path_variant_genes & (cancer_genes | whitelist)

In [ ]:
def test_cancer_gene(df,gene,verbose=False):
    # ecDNA+ gene+ sample count
    a = ((df['amplicon_class']=='ecDNA') & (df['SYMBOL'].map(lambda s: gene in s))).sum()
    # ecDNA+ gene-
    b = ((df['amplicon_class']=='ecDNA') & (df['SYMBOL'].map(lambda s: not gene in s))).sum()
    # ecDNA- gene+
    c = ((df['amplicon_class']!='ecDNA') & (df['SYMBOL'].map(lambda s: gene in s))).sum()
    # ecDNA- gene-
    d = ((df['amplicon_class']!='ecDNA') & (df['SYMBOL'].map(lambda s: not gene in s))).sum()
    if verbose:
        print(gene, a,b,c,d, a+b+c+d)
    if a + c <11:
        return None
    res = scipy.stats.chi2_contingency([[a,b],[c,d]])
    chi2, p, dof, expected = res
    # code for over/underrepresentation
    #YOUR CODE HERE
    if b*c == 0:
        a += 0.5
        b += 0.5
        c += 0.5
        d += 0.5
    odds_ratio = (a*d) / (b*c)
    enriched = True if odds_ratio >1 else False
    return (chi2, p, enriched)
    #return (gene, a,b,c,d, a+b+c+d)

def test_tt_gene(df,gene):
    ct_df = df[df.SYMBOL.map(lambda x: gene in x)].cancer_type.unique()
    df = df[df.cancer_type.isin(ct_df)]
    return test_cancer_gene(df,gene)

def test_tt_all_cancer_gene(df):
    results = []
    for gene in test_genes:
        res = test_tt_gene(df,gene)
        if res is not None:
            chi2, p, enriched = res
            results.append([gene, chi2, p, enriched])
    result_df = pd.DataFrame(results,columns=['gene', 'chi2', 'pvalue', 'enriched'])
    # multiple hypothesis (FDR) correction
    pvals = result_df['pvalue'].values
    reject,pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')

    result_df['FDR'] = pvals_corrected
    result_df['significant(FDR<0.05)'] = reject

    return result_df

In [ ]:
improved_df = test_tt_all_cancer_gene(df)
#improved_df[improved_df['gene'] == 'TP53']
improved_df

In [ ]:
improved_df[improved_df['significant(FDR<0.05)']==True]

In [ ]:
improved_df[improved_df['gene'] == 'TP53']

In [ ]:
target_group = df[
    (df['amplicon_class'] != 'ecDNA') & 
    (df['SYMBOL'].map(lambda s: 'TP53' in s))
]
target_group['cancer_type'].value_counts()

In [ ]:
def stratify_ecDNA_by_tumor_type(df):
    df = df.loc[:,['amplicon_class','cancer_type']]
    df = df.groupby(['cancer_type','amplicon_class']).size()
    return df

In [ ]:
stratify_ecDNA_by_tumor_type(df)

In [ ]:
df_MBL = df[df['cancer_type'].isin(['MBL'])].copy()
df_HGG = df[df['cancer_type'].isin(['HGG'])].copy()

In [ ]:
test_tt_all_cancer_gene(df_MBL)

In [ ]:
test_tt_all_cancer_gene(df_HGG)[test_tt_all_cancer_gene(df_HGG)['significant(FDR<0.05)']==True]

## DEBUG

In [ ]:
big_df = pd.DataFrame()
i=0
dry_run=True
for vcf in get_vcfs():
    i=i+1
    name = extract_sample_name(vcf)
    print('Now processing sample number',i,':',name)

    variants_df = parse_vcf(vcf)
    variants_df['sample'] = name
    big_df = pd.concat([big_df,variants_df])
    if dry_run == True:
        if i > 5:
            break
# one row per gene affected
big_df = big_df.explode('SYMBOL')

In [ ]:
big_df